# 9.1 Publication Text Analysis - step 1: explore "subject" in the data

The ambition of the text analysis is to answer the question - what kinds of research are associated with high vs low energy consumption groups?

The first step is to check the "subject" field to understand whether this gives us useful info about the the kinds of research alone, without exploring the title and abstract.

In [1]:
# Set up
import pandas as pd
import re
import sys
from pathlib import Path

CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

pd.set_option("display.width", 160)

In [2]:
# Load data
publications = pd.read_csv(config.PUBLICATON_DATA / "2_Processed" / "publications_matched.csv")

In [3]:
# No. pubs and distinct labs
print(f"Total publications: {len(publications):,}")
print(f"Distinct labs matched: {publications['matched_labgroupids'].astype(str).str.split(r'[;,]').explode().str.strip().nunique()}")

Total publications: 8,014
Distinct labs matched: 95


In [4]:
# Check what subject looks like
for s in publications["subject"].dropna().sample(15, random_state=1):
    print("-", s[:200])

- 530 Physics
- 910 Geography & travel
- 530 Physics
- 910 Geography & travel
- 570 Life sciences; biology
- 540 Chemistry
- 610 Medicine & health
- General Medicine | General Chemistry | 540 Chemistry
- 610 Medicine & health
- Analytical Chemistry | 540 Chemistry
- 530 Physics
- 610 Medicine & health | 570 Life sciences; biology
- 510 Mathematics | 910 Geography & travel
- 540 Chemistry
- 530 Physics


In [5]:
# Check how many subject tags are classification codes vs free-text keywords

CODE_PATTERN = re.compile(r"^\d{3}\s") # identify classification codes (begin with 3 digits)

subj_tags = publications["subject"].dropna().str.split("|") # subject tags are separated by "|"
subj_flat = subj_tags.explode().str.strip() # flatten to single series of subject tags
is_code = subj_flat.str.match(CODE_PATTERN) # classification code indicator

print(f"Total subject tags across all matched pubs: {len(subj_flat):,}")
print(f"Average tags per publication:                {subj_tags.apply(len).mean():.1f}")
print(f"Share of tags that are classification codes:  {is_code.mean():.0%}")
print(f"Share of tags that are free-text keywords:     {(~is_code).mean():.0%}")

Total subject tags across all matched pubs: 16,149
Average tags per publication:                2.0
Share of tags that are classification codes:  66%
Share of tags that are free-text keywords:     34%


In [12]:
print("Most common classification codes:")
display(subj_flat[is_code].value_counts().head(20))

Most common classification codes:


subject
570 Life sciences; biology                       2970
610 Medicine & health                            2569
530 Physics                                      2259
910 Geography & travel                            617
580 Plants (Botany)                               564
540 Chemistry                                     534
590 Animals (Zoology)                             452
560 Fossils & prehistoric life                    230
510 Mathematics                                   132
300 Social sciences, sociology & anthropology      56
410 Linguistics                                    41
000 Computer science, knowledge & systems          36
340 Law                                            31
490 Other languages                                30
890 Other literatures                              29
170 Ethics                                         15
630 Agriculture                                    13
950 History of Asia                                10
180 Ancient, medieva

In [13]:
print("Most common free-text keywords:")
display(subj_flat[~is_code].value_counts().head(20))

Most common free-text keywords:


subject
Ecology                                         250
Evolution                                       175
Behavior and Systematics                        170
General Chemistry                               129
General Medicine                                100
Genetics and Molecular Biology                   92
General Biochemistry                             90
Multidisciplinary                                84
Nuclear and High Energy Physics                  82
Plant Science                                    79
General Physics and Astronomy                    79
Biochemistry                                     66
Astronomy and Astrophysics                       62
Space and Planetary Science                      59
Molecular Biology                                52
Catalysis                                        52
Genetics                                         44
General Agricultural and Biological Sciences     42
Animal Science and Zoology                       41
Gene

The classification codes are Dewey 2nd level codes - quite coarse. The free-text keywords are sometimes more granular e.g. "catalysis", but can also be quite generic e.g. "general chemistry".

In [14]:
# Check whether pubs have only classification codes or also free-text keywords
def only_codes(tags):
    tags = [t.strip() for t in tags]
    return all(CODE_PATTERN.match(t) for t in tags)

# Number of classification codes and free-text keywords per pub
n_codes = subj_tags.apply(lambda tags: sum(1 for t in tags if CODE_PATTERN.match(t.strip())))
n_free = subj_tags.apply(lambda tags: sum(1 for t in tags if not CODE_PATTERN.match(t.strip())))

print(f"Share of pubs where subject is only classification codes: {subj_tags.apply(only_codes).mean():.0%}")
print(f"Average classification codes per pub: {n_codes.mean():.2f}")
print(f"Average free keywords per pub:        {n_free.mean():.2f}")
print(f"Median free keywords per pub:         {n_free.median():.0f}")
print(f"Unique free-text keyword vocabulary:    {subj_flat[~is_code].str.lower().str.strip().nunique():,}")

Share of pubs where subject is only classification codes: 80%
Average classification codes per pub: 1.33
Average free keywords per pub:        0.69
Median free keywords per pub:         0
Unique free-text keyword vocabulary:    2,122


Only 20% of pubs have any free-text keywords. Most pubs have no free keywords. So subject is a coarse representation of the kind of research that pub represents. This motivates using the title and abstract to measure the kind of research, rather than just the subject. We will use subject as a benchmark measure. 